# Unit 4: 模型训练与优化

## 学习目标
- 掌握完整的模型训练流程
- 理解损失函数与优化器的选择
- 学会实现训练循环
- 掌握学习率调度策略
- 学会数据加载与预处理
- 掌握模型保存与加载方法

## 参考资源
- [PyTorch官方文档 - Optim](https://pytorch.org/docs/stable/optim.html)
- [PyTorch官方文档 - Loss Functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
- [PyTorch教程 - Training a Classifier](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)

## 4.1 损失函数

损失函数衡量模型预测与真实标签之间的差距。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

print("=" * 60)
print("4.1 常用损失函数")
print("=" * 60)

logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))

ce_loss = nn.CrossEntropyLoss()
loss_ce = ce_loss(logits, targets)
print(f"交叉熵损失: {loss_ce.item():.4f}")
print(f"  适用于: 多分类问题")
print(f"  输入: logits(未归一化的预测值), 类别索引")

predictions = torch.randn(4, 1)
targets_reg = torch.randn(4, 1)

mse_loss = nn.MSELoss()
loss_mse = mse_loss(predictions, targets_reg)
print(f"\n均方误差损失: {loss_mse.item():.4f}")
print(f"  适用于: 回归问题")

l1_loss = nn.L1Loss()
loss_l1 = l1_loss(predictions, targets_reg)
print(f"\nL1损失: {loss_l1.item():.4f}")
print(f"  适用于: 回归问题, 对异常值更鲁棒")

binary_logits = torch.randn(4)
binary_targets = torch.randint(0, 2, (4,)).float()

bce_loss = nn.BCEWithLogitsLoss()
loss_bce = bce_loss(binary_logits, binary_targets)
print(f"\n二元交叉熵损失: {loss_bce.item():.4f}")
print(f"  适用于: 二分类问题")

## 4.2 优化器

优化器负责根据梯度更新模型参数。

In [ ]:
print("=" * 60)
print("4.2 常用优化器")
print("=" * 60)

model = nn.Linear(10, 1)

optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
print(f"SGD优化器: lr=0.01, momentum=0.9")
print(f"  特点: 简单, 动量加速收敛")

optimizer_adam = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))
print(f"\nAdam优化器: lr=0.001")
print(f"  特点: 自适应学习率, 最常用")

optimizer_adamw = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
print(f"\nAdamW优化器: lr=0.001, weight_decay=0.01")
print(f"  特点: 解耦权重衰减, 更好的泛化")

optimizer_rmsprop = optim.RMSprop(model.parameters(), lr=0.01, alpha=0.99)
print(f"\nRMSprop优化器: lr=0.01")
print(f"  特点: 适合RNN, 自适应学习率")

## 4.3 数据加载与预处理

In [ ]:
print("=" * 60)
print("4.3 创建模拟数据集")
print("=" * 60)

torch.manual_seed(42)

num_samples = 1000
X_train = torch.randn(num_samples, 3, 32, 32)
y_train = torch.randint(0, 10, (num_samples,))

X_val = torch.randn(200, 3, 32, 32)
y_val = torch.randint(0, 10, (200,))

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(val_dataset)}")
print(f"批次大小: {batch_size}")
print(f"训练批次数: {len(train_loader)}")

for batch_X, batch_y in train_loader:
    print(f"\n单个批次:")
    print(f"  输入形状: {batch_X.shape}")
    print(f"  标签形状: {batch_y.shape}")
    break

## 4.4 定义模型

In [ ]:
print("=" * 60)
print("4.4 定义训练用CNN模型")
print("=" * 60)

class TrainingCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(TrainingCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = TrainingCNN(num_classes=10)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params:,}")

## 4.5 学习率调度策略

In [ ]:
print("=" * 60)
print("4.5 学习率调度器")
print("=" * 60)

model = TrainingCNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler_step = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
print("StepLR: 每10个epoch学习率乘以0.1")

scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
print("CosineAnnealingLR: 余弦退火调度")

scheduler_reduce = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
print("ReduceLROnPlateau: 验证损失不下降时降低学习率")

print("\n学习率变化示例(StepLR):")
for epoch in range(30):
    current_lr = optimizer.param_groups[0]['lr']
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:2d}: lr = {current_lr:.6f}")
    scheduler_step.step()

## 4.6 完整训练循环实现

In [ ]:
print("=" * 60)
print("4.6 完整训练循环")
print("=" * 60)

from utils import get_device

device = get_device()
print(f"使用设备: {device}")

model = TrainingCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

num_epochs = 10
train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_val_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        
        optimizer.step()
        
        running_loss += loss.item() * batch_X.size(0)
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = outputs.max(1)
            val_total += batch_y.size(0)
            val_correct += predicted.eq(batch_y).sum().item()
    
    val_epoch_loss = val_loss / val_total
    val_epoch_acc = 100.0 * val_correct / val_total
    val_losses.append(val_epoch_loss)
    val_accs.append(val_epoch_acc)
    
    scheduler.step()
    
    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.2f}% | "
          f"LR: {current_lr:.6f}")

print(f"\n训练完成! 最佳验证准确率: {best_val_acc:.2f}%")

## 4.7 模型保存与加载

In [ ]:
print("=" * 60)
print("4.7 模型保存与加载")
print("=" * 60)

torch.save({
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_losses[-1],
    'val_acc': val_accs[-1],
}, 'checkpoint.pth')

print("检查点已保存至 checkpoint.pth")

checkpoint = torch.load('checkpoint.pth')
print(f"\n加载的检查点信息:")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Train Loss: {checkpoint['train_loss']:.4f}")
print(f"  Val Acc: {checkpoint['val_acc']:.2f}%")

new_model = TrainingCNN(num_classes=10)
new_model.load_state_dict(checkpoint['model_state_dict'])
print("\n模型权重已成功加载")

new_model.eval()
with torch.no_grad():
    test_input = torch.randn(1, 3, 32, 32)
    output = new_model(test_input)
    print(f"加载后模型测试输出形状: {output.shape}")

## 4.8 训练曲线可视化

In [ ]:
import matplotlib.pyplot as plt

print("=" * 60)
print("4.8 训练曲线可视化")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, num_epochs + 1)
ax1.plot(epochs_range, train_losses, 'b-o', label='Train Loss', linewidth=2, markersize=6)
ax1.plot(epochs_range, val_losses, 'r-o', label='Val Loss', linewidth=2, markersize=6)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, train_accs, 'b-o', label='Train Acc', linewidth=2, markersize=6)
ax2.plot(epochs_range, val_accs, 'r-o', label='Val Acc', linewidth=2, markersize=6)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("训练曲线已绘制")

## 本章小结

本单元我们学习了：
1. 常用损失函数(交叉熵、MSE、L1、BCE)
2. 常用优化器(SGD、Adam、AdamW、RMSprop)
3. 数据加载与DataLoader的使用
4. 学习率调度策略
5. 完整的训练循环实现
6. 模型保存与加载
7. 训练曲线可视化

## 练习建议
1. 尝试不同的优化器和学习率
2. 比较不同学习率调度策略的效果
3. 实验不同的batch size对训练的影响
4. 实现早停法(Early Stopping)

## 下一步
进入Unit 5，学习模型评估与可视化方法。